# Fraud Detection Model
## IEEE-CIS Fraud Detection Kaggle Competition
In this notebook, I built a robust fraud detection pipeline that uses multiple machine learning models to predict fraudulent transactions. The pipeline includes feature engineering, model training, and evaluation.
The models used are:
- LightGBM
- XGBoost
- CatBoost

Where the final ensemble predictions for the test set have an AUC of 0.94311 for public score and 0.92139 for private score, making it one of the top submissions in the competition.

### Dataset Overview

The dataset consists of two tables **transactions** and **identity** linked by `TransactionID`, provided separately for train and test. This results in four files:

- Train transactions  
- Train identity  
- Test transactions  
- Test identity  

Not all transactions have corresponding identity information.

---

#### Transaction Data

- **TransactionID**: Unique transaction identifier  
- **TransactionDT**: Time delta from a reference point (not a real timestamp)  
- **TransactionAmt**: Transaction amount in USD  
- **ProductCD** *(categorical)*: Product code  
- **card1–card6** *(categorical)*: Card-related attributes (e.g. type, country)  
- **addr1, addr2** *(categorical)*: Address information  
- **dist1, dist2**: Distance-related features  
- **P_emaildomain / R_emaildomain** *(categorical)*: Purchaser and recipient email domains  
- **C1–C14**: Count-based features (exact meaning masked)  
- **D1–D15**: Time-delta features between transactions  
- **M1–M9** *(categorical)*: Match indicators (e.g. card vs address)  
- **Vxxx**: Engineered features capturing counts, rankings, and entity relationships  

---

#### Identity Data

- **TransactionID**: Transaction identifier  
- **DeviceType** *(categorical)*: Type of device used  
- **DeviceInfo** *(categorical)*: Detailed device information  
- **id_1–id_38**: Network and browser-related features  
  - *id_12–id_38 are categorical*

---
> **Note:** Column semantics are intentionally masked for security reasons, as this dataset contains real transaction data.


## 1. Data Loading & Memory Reduction

In [31]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import multiprocessing
import lightgbm as lgb
import catboost as cb
from sklearn.metrics import roc_auc_score
from pathlib import Path
import warnings
from pandas.errors import PerformanceWarning

pd.set_option('display.max_columns', 500)
# Ignore warnings for making the outputs clearer
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path.cwd()
DATA_PATH = NOTEBOOK_DIR / "data"
sys.path.append(str(NOTEBOOK_DIR))
# Set random state for reproducibility
RANDOM_STATE = 42

The reduce_mem_usage function is used to reduce the memory usage of the dataframes. Pandas assigns a default data type to each column, which can be memory-intensive. By using reduce_mem_usage, we ensure that the data types are optimized for memory usage.

In [32]:
def reduce_mem_usage(df, verbose=True):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f'Memory usage decreased to {end_mem:.2f} MB '
              f'({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')

    return df

#### "V" Columns Reduction
The following functions from `reduce_correlated_columns.py` groups "V" columns by their NaN structure, identifies highly correlated subsets within each group, and keeps only one representative column from each correlated subset.

The reduction process follows these steps:

1. **Group by NaN Structure**: Columns with similar NaN counts (within tolerance) are grouped together
2. **Find Correlations**: Within each NaN group, identify subsets of columns with correlation > 0.75
3. **Select Representatives**: Keep only the first column from each correlated subset

In [33]:
from scripts.reduce_correlated_columns import reduce_columns_by_correlation

In [34]:
def reduce_v_columns(train_df, test_df):
    """
    Keep only specified V columns or reduce them by correlation.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training dataframe
    test_df : pd.DataFrame
        Test dataframe
    v_cols_to_keep : list of int
        Specific V-column indices to keep. If None, uses correlation reduction.

    Returns
    -------
    tuple of (pd.DataFrame, pd.DataFrame)
        Training and test dataframes with selected V columns
    """
    v_cols = [c for c in train_df.columns if c.startswith('V')]
    print(f"Found {len(v_cols)} V columns initially")

    v_cols_to_keep, v_analysis = reduce_columns_by_correlation(
        train,
        column_prefix='V',
        correlation_threshold=0.75,
        nan_tolerance=100,  # Larger tolerance for V columns
        verbose=True
    )

    print(f"Keeping {len(v_cols_to_keep)} V columns:")
    print(v_cols_to_keep)

    v_cols_to_drop = [c for c in v_cols if c not in v_cols_to_keep]

    train_df = train_df.drop(v_cols_to_drop, axis=1)
    test_df = test_df.drop(v_cols_to_drop, axis=1)

    return train_df, test_df

Here, the key insight is that the challenge is **flagging clients, not individual transactions**.  

Adversarial validation is high (AUC=1) because **clients change over time**, not because fraud patterns shift.  
The model’s goal is to generalize to **unseen clients**, rather than predict based on time.


## 2. Feature Engineering
We first need to identify clients by assigning them a unique identifier (UID), which then allows us to compute client-level aggregations.

One way to approximate a client UID is by leveraging the feature D1, which represents the number of days since a client’s first transaction, together with TransactionDT, the transaction timestamp. By defining a new feature D1n = TransactionDT − D1, we recover the estimated timestamp of the client’s first transaction. Since this value is constant for all transactions of the same client, it can be used as a proxy unique identifier for the client.

In [35]:
def create_normalized_date_features(df):
    """
    Create normalized date features D1n, D4n, D10n, D15n.
    """
    # Calculate current transaction day (day index)
    day_index = np.floor(df['TransactionDT'] / (24 * 60 * 60))

    # Create normalized dates (subtract time difference to get calendar date)
    df['D1n'] = day_index - df['D1']

    # Also for D columns
    for d_col in ['D4', 'D10', 'D15']:
        if d_col in df.columns:
            df[f'{d_col}n'] = day_index - df[d_col]

    return df

Then I constructed a unique user identifier (UID) string by combining card1 + addr1 + D1n. 

Constructing a reliable UID allows us to group transactions by client. If we know a client's history, we can spot anomalies (e.g., a client who usually spends $50 suddenly spending $5000).

This is a critical feature for fraud detection:
- card1: Primary card identifier
- addr1: Billing address
- D1n: Client start date (calendar date)

Same UID = same cardholder. Multiple transactions per UID is normal.

BUT: if UID has inconsistent patterns (varying amounts, dates, etc.),
it may indicate the card was stolen or shared.

In [36]:
def create_uid(df):
    """
    Create UID from card1, addr1, and D1n (start date)

    Returns:
    --------
    Series with UID strings
    """
    # D1n should already be calculated
    if 'D1n' not in df.columns:
        raise ValueError("D1n must be calculated before creating UID")

    # Create UID from card1, addr1, D1n
    uid = (df['card1'].fillna(-999).astype(str) + '_' +
           df['addr1'].fillna(-999).astype(str) + '_' +
           df['D1n'].fillna(-999).astype(str))

    return uid

In [37]:
def encode_aggregations(df, agg_cols, group_cols, agg_funcs, prefix=''):
    """
    Create aggregated features by grouping data.

    This creates features like "mean transaction amount per UID" or
    "std of D4n per UID" which help with data quality and fraud detection:
    - std=0 for date features → single consistent client (UID is correct)
    - std>0 for date features → multiple clients merged under one UID (needs splitting, done by the ML model)
    """
    for col in agg_cols:
        for func in agg_funcs:
            # Compute aggregation
            grouped = df.groupby(group_cols)[col].agg(func)

            # Create descriptive feature name
            group_str = '_'.join(group_cols)
            if prefix:
                feature_name = f'{prefix}_{group_str}_{col}_{func}'
            else:
                feature_name = f'{group_str}_{col}_{func}'

            # Map aggregation back to original dataframe
            if len(group_cols) == 1:
                df[feature_name] = df[group_cols[0]].map(grouped)
            else:
                df[feature_name] = df.set_index(group_cols).index.map(grouped)

    return df


Create hour and day of week features from TransactionDT.

Fraud patterns often vary by time of day and day of week.

In [38]:
def create_time_features(df):
    """
    Create time-based features from TransactionDT.
    """
    df['hour'] = (df['TransactionDT'] // 3600) % 24
    df['day'] = (df['TransactionDT'] // (3600 * 24)) % 7
    return df


The following function extracts the decimal part of the transaction amount (e.g., 49.99 -> 0.99). 

Human transactions often end in .99 or .00. Generated/automated fraud transactions might have random decimal values (e.g., .412) or specific patterns.

In [39]:
def create_cents_feature(df):
    """
    Creates a new feature 'cents' by extracting the decimal part of 'TransactionAmt'.
    """
    df['cents'] = (df['TransactionAmt'] - np.floor(df['TransactionAmt'])).astype('float32')
    return df

Frequency encoding - encode features by their frequency in combined train+test.

This captures how common/rare a value is, which is informative for fraud:
- Very rare card numbers might be more suspicious
- Very common addresses might be less suspicious

In [40]:
def encode_FE(df1, df2, cols):
    """
    Frequency encode specified columns in both dataframes.
    It combines train and test and then encodes

    Parameters
    ----------
    df1 : train dataframe
    df2 : test dataframe
    cols : list of str
        Column names to frequency encode

    Returns
    -------
    tuple of (pd.DataFrame, pd.DataFrame)
        Both dataframes with _FE features added
    """
    for col in cols:
        if col not in df1.columns or col not in df2.columns:
            print(f"Warning: {col} not found in both dataframes, skipping...")
            continue

        # Combine train and test to get overall frequency
        df_combined = pd.concat([df1[col], df2[col]])
        vc = df_combined.value_counts(dropna=True, normalize=True).to_dict()
        vc[-1] = -1  # Handle missing values

        feature_name = col + '_FE'
        df1[feature_name] = df1[col].map(vc).astype('float32')
        df2[feature_name] = df2[col].map(vc).astype('float32')

        # Fill NaN with -1 for missing values
        df1[feature_name].fillna(-1, inplace=True)
        df2[feature_name].fillna(-1, inplace=True)

        print(f"  Created: {feature_name}")

    return df1, df2


Combinatorial Features: Concatenates two columns (e.g., card1 + addr1) and label encodes the result. 

Captures interactions. card1 might be safe, and addr1 might be safe, but that specific combination might be fraudulent.

In [41]:
def encode_CB(df1, df2, col1, col2):
    """
    Combine two features into a single feature, then label encode it.

    Example: card1='1234', addr1='567' -> card1_addr1='1234_567'

    This creates interaction features that capture joint patterns.

    Parameters
    ----------
    df1 : pd.DataFrame (train)
    df2 : pd.DataFrame (test)
    col1 : str
        First column name
    col2 : str
        Second column name

    Returns
    -------
    tuple of (pd.DataFrame, pd.DataFrame)
        Both dataframes with combined feature added
    """
    if col1 not in df1.columns or col2 not in df1.columns:
        print(f"Warning: {col1} or {col2} not found, skipping combination...")
        return df1, df2

    df1 = df1.copy()
    df2 = df2.copy()

    feature_name = f'{col1}_{col2}'

    # Create combined string feature
    df1[feature_name] = df1[col1].astype(str) + '_' + df1[col2].astype(str)
    df2[feature_name] = df2[col1].astype(str) + '_' + df2[col2].astype(str)

    # Label encode the combined feature
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    combined_values = list(df1[feature_name].values) + list(df2[feature_name].values)
    le.fit(combined_values)

    # Transform and convert to int32 separately
    transformed_df1 = le.transform(df1[feature_name])
    transformed_df2 = le.transform(df2[feature_name])

    df1[feature_name] = transformed_df1.astype('int32')
    df2[feature_name] = transformed_df2.astype('int32')

    print(f"  Created: {feature_name}")

    return df1, df2

Function "encode_AG2" calculates the number of unique values (nunique) of one column for each group. 

For example, UID_IP_nunique: How many different IP addresses has this User ID used?

1 IP = Normal.
50 IPs = Very suspicious (likely a bot or stolen account).

In [42]:
def encode_AG2(df1, df2, main_columns, group_columns):
    """
    Create nunique (count unique) aggregation features.

    For each UID, count how many unique values of a feature exist.
    Example: How many unique email domains does this UID have?
    - If nunique=1, it's consistent behavior (same domain always)
    - If nunique>1, might indicate card sharing or fraud

    Parameters
    ----------
    df1 : pd.DataFrame
        First dataframe (typically train)
    df2 : pd.DataFrame
        Second dataframe (typically test)
    main_columns : list of str
        Columns to count unique values for
    group_columns : list of str
        Columns to group by (typically ['UID_encoded'])

    Returns
    -------
    tuple of (pd.DataFrame, pd.DataFrame)
        Both dataframes with nunique features added
    """
    for main_col in main_columns:
        if main_col not in df1.columns or main_col not in df2.columns:
            print(f"Warning: {main_col} not found, skipping nunique aggregation...")
            continue

        for group_col in group_columns:
            if group_col not in df1.columns or group_col not in df2.columns:
                print(f"Warning: {group_col} not found, skipping nunique aggregation...")
                continue

            # Combine train and test for aggregation
            comb = pd.concat([
                df1[[group_col, main_col]],
                df2[[group_col, main_col]]
            ], axis=0)

            # Count unique values per group
            nunique_map = comb.groupby(group_col)[main_col].nunique().to_dict()

            feature_name = f'{group_col}_{main_col}_ct'
            df1[feature_name] = df1[group_col].map(nunique_map).astype('float32')
            df2[feature_name] = df2[group_col].map(nunique_map).astype('float32')

            # Fill NaN with -1
            df1[feature_name].fillna(-1, inplace=True)
            df2[feature_name].fillna(-1, inplace=True)

            print(f"  Created: {feature_name}")

    return df1, df2


## 3. Data Loading and Processing

In [43]:
def load_data(data_path):
    """
    Load and merge transaction and identity data by corresponding TransactionID.

    Parameters
    ----------
    data_path : Path

    Returns
    -------
    tuple of (pd.DataFrame, pd.DataFrame)
        Training and test dataframes
    """
    print("Loading data...")

    try:
        train_identity = pd.read_csv(data_path / "train_identity.csv")
        train_transaction = pd.read_csv(data_path / "train_transaction.csv")
        test_identity = pd.read_csv(data_path / "test_identity.csv")
        test_transaction = pd.read_csv(data_path / "test_transaction.csv")
    except FileNotFoundError as e:
        print(f"Error: Required data file not found - {e}")
        print(f"Expected files in: {data_path}")
        sys.exit(1)

    # Fix column naming inconsistencies in test data (some columns have - instead of _)
    test_identity.columns = test_identity.columns.str.replace('-', '_')

    print("Merging transaction and identity data...")
    train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
    test = pd.merge(test_transaction, test_identity, on='TransactionID', how='left')

    # Clean up memory
    del train_identity, train_transaction, test_identity, test_transaction
    gc.collect()

    return train, test


The "Magic" Wrapper: This function orchestrates the entire feature engineering process in a specific order:

- Normalize D columns: To be used for UID.
- Create Cents: For pattern detection.
- Frequency Encode: For categorical rarity.
- Create Combined Features: For interactions.
- Card Aggregations: Statistics on card usage.
- Create UID: The most critical step.

Magic UID Aggregations: This creates 50+ features grouping by the newly created UID. It answers questions like "Is this transaction amount normal for this specific user?".

Time Features: Hour/Day: 

This is where raw data is transformed into "signal". Without these magic features, the models would only score ~0.90 AUC. With them, they can reach ~0.95+.

In [44]:
def engineer_features(train, test):
    """
    This implements the critical strategies:
    1. D column normalization (transform time deltas to calendar dates)
    2. Cents feature (decimal portion of TransactionAmt)
    3. Frequency encoding for key features
    4. Combined features (card1_addr1, card1_addr1_P_emaildomain)
    5. Card-based aggregations (TransactionAmt, D9, D11)
    6. Magic UID features (47+ features)
    7. Time-based features

    Parameters
    ----------
    train : pd.DataFrame (Training dataframe)
    test : pd.DataFrame (Test dataframe)

    Returns
    -------
    tuple of (pd.DataFrame, pd.DataFrame)
        Training and test dataframes with engineered features
    """
    # STEP 1: NORMALIZE D COLUMNS
    print("\n1. Normalizing D columns (subtract TransactionDT to get calendar dates)...")
    # Skip D1, D2, D3, D5, D9 as per notebook
    d_cols_to_normalize = [4, 6, 7, 8, 10, 11, 12, 13, 14, 15]
    for i in d_cols_to_normalize:
        col_name = f'D{i}'
        if col_name in train.columns:
            train[col_name] = train[col_name] - train['TransactionDT'] / np.float32(24*60*60)
            test[col_name] = test[col_name] - test['TransactionDT'] / np.float32(24*60*60)
            print(f"  Normalized: {col_name}")

    # STEP 2: CREATE CENTS FEATURE
    print("\n2. Creating cents feature (decimal portion of TransactionAmt)...")
    train = create_cents_feature(train)
    test = create_cents_feature(test)
    print("  Created: cents")

    # STEP 3: FREQUENCY ENCODING
    print("\n3. Frequency encoding key features...")
    freq_cols = ['addr1', 'card1', 'card2', 'card3', 'P_emaildomain']
    train, test = encode_FE(train, test, freq_cols)

    # STEP 4: COMBINED FEATURES
    print("\n4. Creating combined features...")
    train, test = encode_CB(train, test, 'card1', 'addr1')
    train, test = encode_CB(train, test, 'card1_addr1', 'P_emaildomain')

    # STEP 5: FREQUENCY ENCODE COMBINED FEATURES
    print("\n5. Frequency encoding combined features...")
    combined_freq_cols = ['card1_addr1', 'card1_addr1_P_emaildomain']
    train, test = encode_FE(train, test, combined_freq_cols)

    # STEP 6: CARD-BASED AGGREGATIONS (NON-UID)
    print("\n6. Creating card-based aggregations...")
    train = train.copy()
    test = test.copy()
    group_features = ['card1', 'card1_addr1', 'card1_addr1_P_emaildomain']
    agg_features = ['TransactionAmt', 'D9', 'D11']
    full = pd.concat([train, test], axis=0)
    for group_col in group_features:
        if group_col not in train.columns:
            continue
        for agg_col in agg_features:
            if agg_col not in train.columns:
                continue
            grouped = full.groupby(group_col)[agg_col]
            mean_map = grouped.mean()
            std_map = grouped.std()
            mean_col = f'{agg_col}_{group_col}_mean'
            std_col  = f'{agg_col}_{group_col}_std'
            train.loc[:, mean_col] = train[group_col].map(mean_map).astype('float32')
            test.loc[:, mean_col]  = test[group_col].map(mean_map).astype('float32')
            train.loc[:, std_col] = train[group_col].map(std_map).astype('float32')
            test.loc[:, std_col]  = test[group_col].map(std_map).astype('float32')
            print(f"  Created: {mean_col}, {std_col}")

    # STEP 7: CREATE D1 for UID (not normalized)
    print("\n7. D1 remains unchanged for UID creation...")
    # D1 is NOT normalized - it's used as-is in the UID calculation
    if 'D1' not in train.columns:
        print("  Warning: D1 not found in data")

    # STEP 8: CREATE UID
    print("\n8. Creating UID (card1_addr1 + floor(day - D1))...")
    # Calculate day index
    train['day'] = train['TransactionDT'] / (24 * 60 * 60)
    test['day'] = test['TransactionDT'] / (24 * 60 * 60)

    # Create UID
    train['UID_encoded'] = (train['card1_addr1'].astype(str) + '_' +
                            np.floor(train['day'] - train['D1']).fillna(-999).astype(str))
    test['UID_encoded'] = (test['card1_addr1'].astype(str) + '_' +
                           np.floor(test['day'] - test['D1']).fillna(-999).astype(str))
    print("  Created: UID_encoded")

    # STEP 9: UID FEATURES (47 FEATURES!)
    print("\n9. Creating UID-based features (this is the secret sauce!)")

    # 9a. Frequency encode UID
    print("  9a. Frequency encoding UID...")
    train, test = encode_FE(train, test, ['UID_encoded'])

    # 9b. Aggregate TransactionAmt, D4, D9, D10, D15 by UID (mean, std)
    print("  9b. Aggregating amounts and D columns by UID...")
    uid_agg_cols = ['TransactionAmt', 'D4', 'D9', 'D10', 'D15']
    for col in uid_agg_cols:
        if col not in train.columns:
            continue
        # Mean
        grouped_mean = pd.concat([train, test]).groupby('UID_encoded')[col].mean()
        train[f'{col}_UID_encoded_mean'] = train['UID_encoded'].map(grouped_mean).astype('float32')
        test[f'{col}_UID_encoded_mean'] = test['UID_encoded'].map(grouped_mean).astype('float32')
        # Std
        grouped_std = pd.concat([train, test]).groupby('UID_encoded')[col].std()
        train[f'{col}_UID_encoded_std'] = train['UID_encoded'].map(grouped_std).astype('float32')
        test[f'{col}_UID_encoded_std'] = test['UID_encoded'].map(grouped_std).astype('float32')
        print(f"    Created: {col}_UID_encoded_mean, {col}_UID_encoded_std")

    # 9c. Aggregate all C columns (except C3) by UID (mean)
    print("  9c. Aggregating C columns by UID...")
    c_cols = [f'C{i}' for i in range(1, 15) if i != 3]
    for col in c_cols:
        if col not in train.columns:
            continue
        grouped_mean = pd.concat([train, test]).groupby('UID_encoded')[col].mean()
        train[f'{col}_UID_encoded_mean'] = train['UID_encoded'].map(grouped_mean).astype('float32')
        test[f'{col}_UID_encoded_mean'] = test['UID_encoded'].map(grouped_mean).astype('float32')
        print(f"    Created: {col}_UID_encoded_mean")

    # 9d. Aggregate all M columns by UID (mean) - but skip M5 as it was removed
    print("  9d. Aggregating M columns by UID...")
    m_cols = [f'M{i}' for i in range(1, 10) if i != 5]  # Skip M5

    # Pre-process M columns to numeric
    m_mapping = {'T': 1, 'F': 0, 'M0': 0, 'M1': 1, 'M2': 2}
    for col in m_cols:
        if col in train.columns:
            # Map values if they are strings
            if train[col].dtype == 'object':
                train[col] = train[col].map(m_mapping)
                print(f"    Mapped {col} to numeric")
        if col in test.columns:
            if test[col].dtype == 'object':
                test[col] = test[col].map(m_mapping)

    for col in m_cols:
        if col not in train.columns:
            continue
        grouped_mean = pd.concat([train, test]).groupby('UID_encoded')[col].mean()
        train[f'{col}_UID_encoded_mean'] = train['UID_encoded'].map(grouped_mean).astype('float32')
        test[f'{col}_UID_encoded_mean'] = test['UID_encoded'].map(grouped_mean).astype('float32')
        print(f"    Created: {col}_UID_encoded_mean")

    # 9e. Count unique values by UID
    print("  9e. Counting unique values by UID...")
    # Create DT_M (month feature) first
    import datetime
    START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')
    train['DT_M'] = train['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds=x)).month)
    test['DT_M'] = test['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds=x)).month)

    nunique_cols = ['P_emaildomain', 'dist1', 'DT_M', 'id_02', 'cents']
    train, test = encode_AG2(train, test, nunique_cols, ['UID_encoded'])

    # 9f. C14 std by UID
    print("  9f. C14 std by UID...")
    if 'C14' in train.columns:
        grouped_std = pd.concat([train, test]).groupby('UID_encoded')['C14'].std()
        train['C14_UID_encoded_std'] = train['UID_encoded'].map(grouped_std).astype('float32')
        test['C14_UID_encoded_std'] = test['UID_encoded'].map(grouped_std).astype('float32')
        print("    Created: C14_UID_encoded_std")

    # 9g. More nunique features
    print("  9g. More nunique counts by UID...")
    more_nunique_cols = ['C13', 'V314']
    train, test = encode_AG2(train, test, more_nunique_cols, ['UID_encoded'])

    v_nunique_cols = ['V95', 'V135', 'V279', 'V309', 'V319']
    train, test = encode_AG2(train, test, v_nunique_cols, ['UID_encoded'])

    # 9h. Outsider15 feature
    print("  9h. Creating outsider15 feature...")
    if 'D1' in train.columns and 'D15' in train.columns:
        train['outsider15'] = (np.abs(train['D1'] - train['D15']) > 3).astype('int8')
        test['outsider15'] = (np.abs(test['D1'] - test['D15']) > 3).astype('int8')
        print("    Created: outsider15")

    # STEP 10: TIME FEATURES
    print("\n10. Creating time-based features (hour, day of week)...")
    train = create_time_features(train)
    test = create_time_features(test)

    print("Feature engineering complete! Added 50+ magic features.")
    print(f"Train shape: {train.shape}, Test shape: {test.shape}")

    return train, test


Now we enconde categorical features. 
This converts string categories into integers using LabelEncoder.

This is done since Machine learning models (especially XGBoost and LightGBM in standard modes) require numerical input.

In [45]:
TRANSACTION_CATEGORICAL = [
    'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain',
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9'
]

IDENTITY_CATEGORICAL = [
    'DeviceType', 'DeviceInfo',
    'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18',
    'id_19', 'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25',
    'id_26', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_32',
    'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38'
]

def encode_categorical_features(train, test):
    """
    Encode categorical features using LabelEncoder.

    Parameters
    ----------
    train : pd.DataFrame (Training dataframe)
    test : pd.DataFrame (Test dataframe)

    Returns
    -------
    tuple of (pd.DataFrame, pd.DataFrame, list)
        Training and test dataframes with encoded features, and list of
        categorical column names that were successfully encoded
    """
    print("Encoding categorical features...")

    categorical_features = TRANSACTION_CATEGORICAL + IDENTITY_CATEGORICAL + ['UID_encoded']
    encoded_categorical = []

    for col in categorical_features:
        if col in train.columns and col in test.columns:
            le = LabelEncoder()
            # Fit on combined train+test to handle unseen labels gracefully
            combined_values = (
                list(train[col].astype(str).values) +
                list(test[col].astype(str).values)
            )
            le.fit(combined_values)

            train[col] = le.transform(list(train[col].astype(str).values))
            test[col] = le.transform(list(test[col].astype(str).values))
            encoded_categorical.append(col)
        else:
            print(f'  Skipping {col} (not found in both train and test)')

    print(f"Encoded {len(encoded_categorical)} categorical features")
    return train, test, encoded_categorical

#### Prepare Model Data

Drops Overfitting Columns: Removes original ID columns that specific to the train set.

Sorts by Time: train = train.sort_values('TransactionDT').

Time-Series Split: Splits the last 20% of data as validation.

Time-Sorting is Non-Negotiable: You cannot do a random K-Fold split in fraud detection. You must train on the past and predict the future. Random splitting would "leak" future fraud patterns into the training set, giving you a falsely high score that fails in production.

In [46]:
COLS_TO_DROP_FOR_MODELING = [
    'TransactionDay',  # Temporary feature
    'D4n', 'D10n', 'D15n',  # Raw features (we keep aggregations)
    'UID_encoded',  # Raw UID (we keep UID-based aggregations)
    'DeviceInfo'  # Very high cardinality
]

# Train/validation split ratio (time-based split)
TRAIN_VAL_SPLIT = 0.75

def prepare_model_data(train, test):
    """
    Prepare final datasets for modeling.

    Steps:
    1. Drop temporary/identifier columns
    2. Sort by time for proper train/val split
    3. Create train/validation split (time-based)

    Parameters
    ----------
    train : pd.DataFrame
    test : pd.DataFrame

    Returns
    -------
    tuple of (X_train, X_val, y_train, y_val, X_test)
        Train/validation/test splits ready for modeling
    """
    print("PREPARING DATA FOR MODELING")

    # Calculate temporary day index for sorting (then drop it)
    train['TransactionDay'] = train['TransactionDT'] / (24 * 60 * 60)

    # Drop identification columns to prevent overfitting
    print("\nDropping identifier columns to prevent overfitting...")
    for col in COLS_TO_DROP_FOR_MODELING:
        if col in train.columns:
            train.drop(col, axis=1, inplace=True)
            print(f"  Dropped: {col}")
        if col in test.columns:
            test.drop(col, axis=1, inplace=True)

    # Sort by time (critical for time-based split)
    print("\nSorting by TransactionDT for time-based validation...")
    train = train.sort_values('TransactionDT')

    # Create features (X) and target (y)
    drop_cols = ['isFraud', 'TransactionDT', 'TransactionID']
    X = train.drop(drop_cols, axis=1)
    y = train['isFraud']

    # Prepare test data (only drop columns that exist)
    X_test = test.drop([c for c in drop_cols if c in test.columns], axis=1)

    # Ensure train and test have same columns
    X_test = X_test[X.columns]

    # Time-based train/validation split (80/20)
    split_idx = int(TRAIN_VAL_SPLIT * len(X))
    X_train = X.iloc[:split_idx]
    X_val = X.iloc[split_idx:]
    y_train = y.iloc[:split_idx]
    y_val = y.iloc[split_idx:]

    print(f"\nTrain shape: {X_train.shape}")
    print(f"Validation shape: {X_val.shape}")
    print(f"Test shape: {X_test.shape}")
    print(f"\nFraud rate in train: {y_train.mean():.4f}")
    print(f"Fraud rate in validation: {y_val.mean():.4f}")

    return X_train, X_val, y_train, y_val, X_test


## 4. Model Training

In [47]:
# General configuration for model training

# Early stopping rounds
EARLY_STOPPING_ROUNDS = 200

# Logging frequency during training
LOG_EVAL_PERIOD = 100

In [48]:
CATBOOST_PARAMS = {
    'iterations': 2000,
    'depth': 8,
    'learning_rate': 0.02,
    'l2_leaf_reg': 5,
    'eval_metric': 'AUC',
    'random_seed': RANDOM_STATE,
    'bagging_temperature': 0.5,
    'od_type': 'Iter',
    'metric_period': 100,
    'od_wait': 100,
    'allow_writing_files': False,
    # 'grow_policy': 'Lossguide',
    'num_leaves': 256,
    'bootstrap_type': 'MVS',
    'subsample': 0.8,
    'rsm': 0.7,
    'border_count': 254,
    'task_type': 'CPU'
}

def train_catboost(X_train, y_train, X_val, y_val, categorical_features):
    """
    Train CatBoost classifier with early stopping.

    Parameters
    ----------
    X_train : pd.DataFrame
        Training features
    y_train : pd.Series
        Training labels
    X_val : pd.DataFrame
        Validation features
    y_val : pd.Series
        Validation labels
    categorical_features : list
        List of categorical feature names

    Returns
    -------
    cb.CatBoostClassifier
        Trained CatBoost model
    """

    print("TRAINING CATBOOST MODEL")
    print(f"\nHyperparameters:")
    for key, value in CATBOOST_PARAMS.items():
        print(f"  {key}: {value}")

    # Initialize model
    clf = cb.CatBoostClassifier(**CATBOOST_PARAMS)

    # Train with early stopping
    clf.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=[c for c in categorical_features if c in X_train.columns],
        use_best_model=True,
        verbose=LOG_EVAL_PERIOD
    )

    # Calculate and display validation metrics
    val_preds = clf.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_preds)

    print(f"VALIDATION AUC (CatBoost): {val_auc:.6f}")
    print(f"Best iteration: {clf.get_best_iteration()}")

    return clf


In [49]:
LGBM_PARAMS = {
    # 'n_estimators': 1000,
    # 'learning_rate': 0.02,
    # 'num_leaves': 128,
    'n_estimators': 5000,
    'learning_rate': 0.01,
    'num_leaves': 256,
    'objective': 'binary',
    'metric': 'auc',
    'min_child_samples': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'random_state': RANDOM_STATE,
    'n_jobs': 4,
    'verbose': -1
}

def train_lightgbm(X_train, y_train, X_val, y_val):
    """
    Train LightGBM classifier with early stopping.

    Parameters
    ----------
    X_train : pd.DataFrame
        Training features
    y_train : pd.Series
        Training labels
    X_val : pd.DataFrame
        Validation features
    y_val : pd.Series
        Validation labels

    Returns
    -------
    lgb.LGBMClassifier
        Trained LightGBM model
    """
    print("TRAINING LIGHTGBM MODEL")
    print(f"\nHyperparameters:")
    for key, value in LGBM_PARAMS.items():
        print(f"  {key}: {value}")
    print(f"  early_stopping_rounds: {EARLY_STOPPING_ROUNDS}")
    print()

    # Initialize model
    clf = lgb.LGBMClassifier(**LGBM_PARAMS)

    # Train with early stopping
    clf.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING_ROUNDS),
            lgb.log_evaluation(LOG_EVAL_PERIOD)
        ]
    )

    # Calculate and display validation metrics
    val_preds = clf.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_preds)

    print(f"VALIDATION AUC: {val_auc:.6f}")
    print(f"Best iteration: {clf.best_iteration_}")

    return clf

In [50]:
XGB_PARAMS = {
    'n_estimators': 2500,
    'learning_rate': 0.02,
    'max_depth': 12,
   #  'n_estimators': 500,
   #  'learning_rate': 0.05,
   #  'max_depth': 4,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'subsample': 0.8,
    'colsample_bytree': 0.4,
    'tree_method': 'hist',
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'early_stopping_rounds': 100
}

def train_xgboost(X_train, y_train, X_val, y_val):
    """
    Train XGBoost classifier with early stopping.

    Parameters
    ----------
    X_train : pd.DataFrame
       Training features
    y_train : pd.Series
       Training labels
    X_val : pd.DataFrame
       Validation features
    y_val : pd.Series
       Validation labels

    Returns
    -------
    xgb.XGBClassifier
       Trained XGBoost model
    """
    print("TRAINING XGBOOST MODEL")
    print(f"\nHyperparameters:")
    for key, value in XGB_PARAMS.items():
        print(f"  {key}: {value}")

    # Initialize model
    clf = xgb.XGBClassifier(**XGB_PARAMS)
    # Note: Using XGB_PARAMS from config

    # Train with early stopping
    clf.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        # eval_metric='auc', # Already in XGB_PARAMS
        verbose=LOG_EVAL_PERIOD
    )

    # Calculate and display validation metrics
    val_preds = clf.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_preds)

    print(f"VALIDATION AUC (XGB): {val_auc:.6f}")
    if hasattr(clf, 'best_iteration'):
        print(f"Best iteration: {clf.best_iteration}")

    return clf


In [ ]:
def create_submission(predictions, data_path, filename="submission.csv"):
    """
    Create submission file with fraud predictions.

    Parameters
    ----------
    predictions : np.array
        Probability predictions
    data_path : Path
        Path to data directory
    filename : str, default="submission.csv"
        Name of the output file

    Returns
    -------
    None
        Saves submission.csv to parent directory
    """
    print(f"Creating submission file: {filename}...")

    # Load sample submission and update with predictions
    submission_df = pd.read_csv(data_path / "sample_submission.csv")
    submission_df['isFraud'] = predictions

    # Save submission
    submission_path = data_path / filename
    submission_df.to_csv(submission_path, index=False)

    print(f"Submission saved to: {submission_path}")
    print(f"Predictions range: [{predictions.min():.4f}, {predictions.max():.4f}]")
    print(f"Mean prediction: {predictions.mean():.4f}")

## 5. Main Model Execution

In [52]:
# 1. Load data
train, test = load_data(DATA_PATH)

Loading data...
Merging transaction and identity data...


In [53]:
# 2. Reduce memory usage
print("\nOptimizing memory usage...")
train = reduce_mem_usage(train)
test = reduce_mem_usage(test)


Optimizing memory usage...
Memory usage decreased to 1044.70 MB (46.6% reduction)
Memory usage decreased to 895.89 MB (46.5% reduction)


In [54]:
# 3. Reduce V columns (user specified list)
print(f"\nSelecting V columns from specified list...")
train, test = reduce_v_columns(train, test)


Selecting V columns from specified list...
Found 339 V columns initially

Analyzing V columns: 339 total

NaN Group (≈0 NaNs): 32 columns
  Columns: ['V279', 'V280', 'V284', 'V285', 'V286', 'V287', 'V290', 'V291', 'V292', 'V293', 'V294', 'V295', 'V297', 'V298', 'V299', 'V302', 'V303', 'V304', 'V305', 'V306', 'V307', 'V308', 'V309', 'V310', 'V311', 'V312', 'V316', 'V317', 'V318', 'V319', 'V320', 'V321']
  Correlated subset: ['V279', 'V280', 'V293', 'V294', 'V295', 'V298', 'V299', 'V306', 'V307', 'V308', 'V316', 'V317', 'V318']
    → Keeping: V279
  Correlated subset: ['V285', 'V287']
    → Keeping: V285
  Correlated subset: ['V290', 'V292']
    → Keeping: V290
  Correlated subset: ['V302', 'V303', 'V304']
    → Keeping: V302
  Correlated subset: ['V309', 'V311']
    → Keeping: V309
  Correlated subset: ['V319', 'V320', 'V321']
    → Keeping: V319
  Reduction: 32 → 13 columns

NaN Group (≈300 NaNs): 43 columns
  Columns: ['V95', 'V96', 'V97', 'V98', 'V99', 'V100', 'V101', 'V102', 'V103'

In [55]:
# 4. Feature engineering
train, test = engineer_features(train, test)


1. Normalizing D columns (subtract TransactionDT to get calendar dates)...
  Normalized: D4
  Normalized: D6
  Normalized: D7
  Normalized: D8
  Normalized: D10
  Normalized: D11
  Normalized: D12
  Normalized: D13
  Normalized: D14
  Normalized: D15

2. Creating cents feature (decimal portion of TransactionAmt)...
  Created: cents

3. Frequency encoding key features...
  Created: addr1_FE
  Created: card1_FE
  Created: card2_FE
  Created: card3_FE
  Created: P_emaildomain_FE

4. Creating combined features...
  Created: card1_addr1
  Created: card1_addr1_P_emaildomain

5. Frequency encoding combined features...
  Created: card1_addr1_FE
  Created: card1_addr1_P_emaildomain_FE

6. Creating card-based aggregations...
  Created: TransactionAmt_card1_mean, TransactionAmt_card1_std
  Created: D9_card1_mean, D9_card1_std
  Created: D11_card1_mean, D11_card1_std
  Created: TransactionAmt_card1_addr1_mean, TransactionAmt_card1_addr1_std
  Created: D9_card1_addr1_mean, D9_card1_addr1_std
  Cre

In [56]:
# 5. Encode categorical features
train, test, _ = encode_categorical_features(train, test)

Encoding categorical features...
Encoded 50 categorical features


In [57]:
# 6. Prepare modeling data
X_train, X_val, y_train, y_val, X_test = prepare_model_data(train, test)

# Clean up memory
del train, test
gc.collect()

PREPARING DATA FOR MODELING

Dropping identifier columns to prevent overfitting...
  Dropped: TransactionDay
  Dropped: UID_encoded
  Dropped: DeviceInfo

Sorting by TransactionDT for time-based validation...

Train shape: (442905, 309)
Validation shape: (147635, 309)
Test shape: (506691, 309)

Fraud rate in train: 0.0351
Fraud rate in validation: 0.0345


0

In [58]:
# 7. Train models
print("\nTraining Ensemble Models...")
cat_feats = [c for c in TRANSACTION_CATEGORICAL + IDENTITY_CATEGORICAL + ['UID_encoded'] if c in X_train.columns]
catboost_model = train_catboost(X_train, y_train, X_val, y_val, cat_feats)


Training Ensemble Models...
TRAINING CATBOOST MODEL

Hyperparameters:
  iterations: 2000
  depth: 8
  learning_rate: 0.02
  l2_leaf_reg: 5
  eval_metric: AUC
  random_seed: 42
  bagging_temperature: 0.5
  od_type: Iter
  metric_period: 100
  od_wait: 100
  allow_writing_files: False
  num_leaves: 256
  bootstrap_type: MVS
  subsample: 0.8
  rsm: 0.7
  border_count: 254
  task_type: CPU


0:	test: 0.7048659	best: 0.7048659 (0)	total: 1.19s	remaining: 39m 32s
100:	test: 0.8859957	best: 0.8859957 (100)	total: 1m 46s	remaining: 33m 22s
200:	test: 0.9042490	best: 0.9042490 (200)	total: 3m 45s	remaining: 33m 36s
300:	test: 0.9127545	best: 0.9127615 (299)	total: 5m 37s	remaining: 31m 47s
400:	test: 0.9189692	best: 0.9189692 (400)	total: 7m 32s	remaining: 30m 3s
500:	test: 0.9231134	best: 0.9231275 (498)	total: 9m 27s	remaining: 28m 19s
600:	test: 0.9260485	best: 0.9260485 (600)	total: 11m 27s	remaining: 26m 40s
700:	test: 0.9277051	best: 0.9277994 (683)	total: 13m 20s	remaining: 24m 44s
800:	test: 0.9285192	best: 0.9285219 (796)	total: 15m 19s	remaining: 22m 57s
900:	test: 0.9288437	best: 0.9288626 (857)	total: 17m 9s	remaining: 20m 55s
1000:	test: 0.9298294	best: 0.9298350 (997)	total: 19m 6s	remaining: 19m 4s
1100:	test: 0.9305016	best: 0.9305198 (1093)	total: 21m 7s	remaining: 17m 14s
1200:	test: 0.9312421	best: 0.9312464 (1198)	total: 23m 5s	remaining: 15m 21s
1300:	test:

In [59]:
lgb_model = train_lightgbm(X_train, y_train, X_val, y_val)

TRAINING LIGHTGBM MODEL

Hyperparameters:
  n_estimators: 5000
  learning_rate: 0.01
  num_leaves: 256
  objective: binary
  metric: auc
  min_child_samples: 50
  subsample: 0.8
  colsample_bytree: 0.7
  random_state: 42
  n_jobs: 4
  verbose: -1
  early_stopping_rounds: 200

Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.917219
[200]	valid_0's auc: 0.928821
[300]	valid_0's auc: 0.936991
[400]	valid_0's auc: 0.941949
[500]	valid_0's auc: 0.944509
[600]	valid_0's auc: 0.946073
[700]	valid_0's auc: 0.946703
[800]	valid_0's auc: 0.946868
[900]	valid_0's auc: 0.946858
[1000]	valid_0's auc: 0.946901
[1100]	valid_0's auc: 0.94686
[1200]	valid_0's auc: 0.946436
Early stopping, best iteration is:
[1053]	valid_0's auc: 0.947027
VALIDATION AUC: 0.947027
Best iteration: 1053


In [60]:
xgb_model = train_xgboost(X_train, y_train, X_val, y_val)

TRAINING XGBOOST MODEL

Hyperparameters:
  n_estimators: 2500
  learning_rate: 0.02
  max_depth: 12
  objective: binary:logistic
  eval_metric: auc
  subsample: 0.8
  colsample_bytree: 0.4
  tree_method: hist
  random_state: 42
  n_jobs: -1
  early_stopping_rounds: 100
[0]	validation_0-auc:0.80932
[100]	validation_0-auc:0.93349
[200]	validation_0-auc:0.94291
[300]	validation_0-auc:0.94679
[400]	validation_0-auc:0.94744
[493]	validation_0-auc:0.94738
VALIDATION AUC (XGB): 0.947528
Best iteration: 393


## 6. Ensemble Model 

In [61]:
# 8. Evaluation and Ensembling
print("ENSEMBLE EVALUATION")

cat_val_preds = catboost_model.predict_proba(X_val)[:, 1]
lgb_val_preds = lgb_model.predict_proba(X_val)[:, 1]
xgb_val_preds = xgb_model.predict_proba(X_val)[:, 1]

cat_auc = roc_auc_score(y_val, cat_val_preds)
lgb_auc = roc_auc_score(y_val, lgb_val_preds)
xgb_auc = roc_auc_score(y_val, xgb_val_preds)

print(f"CatBoost Validation AUC: {cat_auc:.6f}")
print(f"LightGBM Validation AUC: {lgb_auc:.6f}")
print(f"XGBoost  Validation AUC: {xgb_auc:.6f}")

ENSEMBLE EVALUATION
CatBoost Validation AUC: 0.935289
LightGBM Validation AUC: 0.947027
XGBoost  Validation AUC: 0.947528


In [62]:
def evaluate_ensemble_methods(y_true, lgb_preds, xgb_preds, cat_preds):
    """Compare different ensemble approaches"""

    results = {}

    # 1. Simple Average
    simple_avg = (lgb_preds + xgb_preds + cat_preds) / 3
    results['Simple Average'] = roc_auc_score(y_true, simple_avg)

    # 2. Weighted by AUC
    weights = np.array([
        roc_auc_score(y_true, lgb_preds),
        roc_auc_score(y_true, xgb_preds),
        roc_auc_score(y_true, cat_preds)
    ])
    weights = weights / weights.sum()
    weighted_avg = (weights[0] * lgb_preds +
                    weights[1] * xgb_preds +
                    weights[2] * cat_preds)
    results['Weighted Average'] = roc_auc_score(y_true, weighted_avg)

    # 3. Rank Average
    from scipy.stats import rankdata
    ranks = np.column_stack([
        rankdata(lgb_preds),
        rankdata(xgb_preds),
        rankdata(cat_preds)
    ]).mean(axis=1)
    results['Rank Average'] = roc_auc_score(y_true, ranks)

    # 4. Stacking
    from sklearn.linear_model import LogisticRegression
    stacked = np.column_stack([lgb_preds, xgb_preds, cat_preds])
    meta_model = LogisticRegression()
    meta_model.fit(stacked, y_true)
    stacked_preds = meta_model.predict_proba(stacked)[:, 1]
    results['Stacking (LR)'] = roc_auc_score(y_true, stacked_preds)

    return results, meta_model

# Evaluation of the ensemble models under different methods
results, best_meta_model = evaluate_ensemble_methods(
    y_val, lgb_val_preds, xgb_val_preds, cat_val_preds
)

for method, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{method:20s}: {score:.6f}")

best_method, best_score = max(results.items(), key=lambda x: x[1])

Weighted Average    : 0.948066
Simple Average      : 0.948045
Rank Average        : 0.948041
Stacking (LR)       : 0.946944


In [63]:
print(f"Best method is {best_method} with an AUC score of {best_score:.4f}")

Best method is Weighted Average with an AUC score of 0.9481


## 7. Final Submission

In [64]:
print("Generating test predictions...")

cat_test_preds = catboost_model.predict_proba(X_test)[:, 1]
lgb_test_preds = lgb_model.predict_proba(X_test)[:, 1]
xgb_test_preds = xgb_model.predict_proba(X_test)[:, 1]

if best_method == "Simple Average":
    ensemble_test_preds = (
        lgb_test_preds + xgb_test_preds + cat_test_preds
    ) / 3

elif best_method == "Weighted Average":
    weights = np.array([
        roc_auc_score(y_val, lgb_val_preds),
        roc_auc_score(y_val, xgb_val_preds),
        roc_auc_score(y_val, cat_val_preds)
    ])
    weights /= weights.sum()

    ensemble_test_preds = (
        weights[0] * lgb_test_preds +
        weights[1] * xgb_test_preds +
        weights[2] * cat_test_preds
    )

elif best_method == "Rank Average":
    from scipy.stats import rankdata

    ensemble_test_preds = np.column_stack([
        rankdata(lgb_test_preds),
        rankdata(xgb_test_preds),
        rankdata(cat_test_preds)
    ]).mean(axis=1)

elif best_method == "Stacking (LR)":
    stacked_test = np.column_stack([
        lgb_test_preds,
        xgb_test_preds,
        cat_test_preds
    ])
    ensemble_test_preds = best_meta_model.predict_proba(stacked_test)[:, 1]

else:
    raise ValueError(f"Unknown ensemble method: {best_method}")


Generating test predictions...


In [67]:
SUBMISSION_PATH = DATA_PATH.parent / "submissions"
create_submission(ensemble_test_preds, SUBMISSION_PATH, filename=f"submission_best_{best_method.replace(' ', '_')}.csv")
create_submission(lgb_test_preds, SUBMISSION_PATH, filename="submission_lgb.csv")
create_submission(xgb_test_preds, SUBMISSION_PATH, filename="submission_xgb.csv")
create_submission(cat_test_preds, SUBMISSION_PATH, filename="submission_cat.csv")

Creating submission file: submission_best_Weighted_Average.csv...
Submission saved to: /Users/felipemediavillalevinson/Documents/Kaggle_Fraud/submission_best_Weighted_Average.csv
Predictions range: [0.0002, 0.9982]
Mean prediction: 0.0284
Creating submission file: submission_lgb.csv...
Submission saved to: /Users/felipemediavillalevinson/Documents/Kaggle_Fraud/submission_lgb.csv
Predictions range: [0.0001, 0.9993]
Mean prediction: 0.0264
Creating submission file: submission_xgb.csv...
Submission saved to: /Users/felipemediavillalevinson/Documents/Kaggle_Fraud/submission_xgb.csv
Predictions range: [0.0001, 0.9970]
Mean prediction: 0.0260
Creating submission file: submission_cat.csv...
Submission saved to: /Users/felipemediavillalevinson/Documents/Kaggle_Fraud/submission_cat.csv
Predictions range: [0.0002, 0.9997]
Mean prediction: 0.0329
